In [2]:
%pip install nrlmsise00

Note: you may need to restart the kernel to use updated packages.


In [61]:

import pandas as pd

url = "https://celestrak.org/SpaceData/SW-All.csv"

data = pd.read_csv(url)

aug15 = data[data["DATE"] == "2026-08-15"].iloc[0]
aug16 = data[data["DATE"] == "2026-08-16"].iloc[0]

f107 = float(aug15["F10.7_OBS"])
f107a = float(aug16["F10.7_OBS_CENTER81"])
ap = float(aug16["AP_AVG"])

print("F107:", f107)
print("F107A:", f107a)
print("Ap:", ap)


F107: 117.1
F107A: 117.1
Ap: 3.0


In [62]:

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

# Constants
D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

# I picked Russellville on august 16th 2026 
date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338


# Atmospheric density and temp
def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

# Peice wise function of drage coeffiecnt 
def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

# Differential equations
def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]


# Initial conditions
y = 0
v = 7599.68
t = 0

# time intervals of 0.5 seconds 
dt = 0.5

# set to end loop when rocket stops moving up
while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.5 s
Altitude: 3.79984 km
Velocity: 5386.475642495565 m/s
--------------------
Time: 1.0 s
Altitude: 6.493077821247782 km
Velocity: 4625.852866707133 m/s
--------------------
Time: 1.5 s
Altitude: 8.80600425460135 km
Velocity: 4196.662780701345 m/s
--------------------
Time: 2.0 s
Altitude: 10.904335644952022 km
Velocity: 3919.2000637695664 m/s
--------------------
Time: 2.5 s
Altitude: 12.863935676836805 km
Velocity: 3727.705410520474 m/s
--------------------
Time: 3.0 s
Altitude: 14.727788382097042 km
Velocity: 3591.230411958114 m/s
--------------------
Time: 3.5 s
Altitude: 16.5234035880761 km
Velocity: 3492.7078292269575 m/s
--------------------
Time: 4.0 s
Altitude: 18.26975750268958 km
Velocity: 3421.1347523619097 m/s
--------------------
Time: 4.5 s
Altitude: 19.980324878870537 km
Velocity: 3368.430650115726 m/s
--------------------
Time: 5.0 s
Altitude: 21.664540203928397 km
Velocity: 3328.7804246191536 m/s
--------------------
Time: 5.5 s
Altitude: 23.328930416237974 km

In [63]:
# Checking V0 of 2000 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]


y = 0
v = 2000
t = 0

dt = 0.5

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.5 s
Altitude: 1.0 km
Velocity: 1871.5647635911844 m/s
--------------------
Time: 1.0 s
Altitude: 1.9357823817955924 km
Velocity: 1773.736841122295 m/s
--------------------
Time: 1.5 s
Altitude: 2.82265080235674 km
Velocity: 1692.9053431104467 m/s
--------------------
Time: 2.0 s
Altitude: 3.6691034739119632 km
Velocity: 1624.706336543756 m/s
--------------------
Time: 2.5 s
Altitude: 4.481456642183841 km
Velocity: 1566.2194877451313 m/s
--------------------
Time: 3.0 s
Altitude: 5.264566386056407 km
Velocity: 1515.402559438432 m/s
--------------------
Time: 3.5 s
Altitude: 6.022267665775622 km
Velocity: 1470.7757675276434 m/s
--------------------
Time: 4.0 s
Altitude: 6.757655549539444 km
Velocity: 1431.2349563080224 m/s
--------------------
Time: 4.5 s
Altitude: 7.473273027693455 km
Velocity: 1395.935643483974 m/s
--------------------
Time: 5.0 s
Altitude: 8.171240849435442 km
Velocity: 1364.2181663756087 m/s
--------------------
Time: 5.5 s
Altitude: 8.853349932623248 km
Velo

In [64]:
# Checking V0 of 2200 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]



y = 0
v = 2200
t = 0

dt = 0.5

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.5 s
Altitude: 1.1 km
Velocity: 2031.3880247972697 m/s
--------------------
Time: 1.0 s
Altitude: 2.115694012398635 km
Velocity: 1912.7034707057956 m/s
--------------------
Time: 1.5 s
Altitude: 3.072045747751533 km
Velocity: 1816.7523668830445 m/s
--------------------
Time: 2.0 s
Altitude: 3.9804219311930553 km
Velocity: 1740.7434038377974 m/s
--------------------
Time: 2.5 s
Altitude: 4.850793633111953 km
Velocity: 1676.2102483410126 m/s
--------------------
Time: 3.0 s
Altitude: 5.68889875728246 km
Velocity: 1620.6332119609433 m/s
--------------------
Time: 3.5 s
Altitude: 6.499215363262931 km
Velocity: 1572.2154321565365 m/s
--------------------
Time: 4.0 s
Altitude: 7.285323079341199 km
Velocity: 1529.6332590220072 m/s
--------------------
Time: 4.5 s
Altitude: 8.050139708852203 km
Velocity: 1491.8848588262038 m/s
--------------------
Time: 5.0 s
Altitude: 8.796082138265305 km
Velocity: 1458.1943353570928 m/s
--------------------
Time: 5.5 s
Altitude: 9.525179305943851 km
V

In [65]:
# Checking V0 of 2100 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]



y = 0
v = 2100
t = 0

dt = 0.5

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.5 s
Altitude: 1.05 km
Velocity: 1945.9322395363552 m/s
--------------------
Time: 1.0 s
Altitude: 2.0229661197681774 km
Velocity: 1836.0863459880106 m/s
--------------------
Time: 1.5 s
Altitude: 2.9410092927621827 km
Velocity: 1750.5319955233667 m/s
--------------------
Time: 2.0 s
Altitude: 3.816275290523866 km
Velocity: 1678.7443747160582 m/s
--------------------
Time: 2.5 s
Altitude: 4.655647477881895 km
Velocity: 1617.4672843343913 m/s
--------------------
Time: 3.0 s
Altitude: 5.46438112004909 km
Velocity: 1564.4448702355564 m/s
--------------------
Time: 3.5 s
Altitude: 6.246603555166868 km
Velocity: 1518.0547098984878 m/s
--------------------
Time: 4.0 s
Altitude: 7.005630910116112 km
Velocity: 1477.093488836231 m/s
--------------------
Time: 4.5 s
Altitude: 7.744177654534228 km
Velocity: 1440.6454046145395 m/s
--------------------
Time: 5.0 s
Altitude: 8.464500356841498 km
Velocity: 1407.997976008027 m/s
--------------------
Time: 5.5 s
Altitude: 9.168499344845511 km
V

In [66]:
# Checking V0 of 2100.5 m/s

import numpy as np
from datetime import datetime
from nrlmsise00 import msise_model

D = 0.5
m = 400
g = 9.81

A = np.pi * (D / 2)**2

date = datetime(2026, 8, 16, 17, 10)
lat = 35.2784
lon = -93.1338

def rho(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon,  f107a, f107, ap
    )
    return densities[5] * 1000

def T(y):
    densities, temperatures = msise_model(
        date, y / 1000, lat, lon, f107a, f107, ap
    )
    return temperatures[1]

def Cd(M):
    if M <= 0.5:
        return 0.26
    elif M <= 1:
        return 0.4
    elif M <= 1.5:
        return 0.55
    elif M <= 2:
        return 0.47
    elif M <= 2.5:
        return 0.36
    elif M <= 3:
        return 0.28
    elif M <= 3.5:
        return 0.24
    elif M <= 4:
        return 0.21
    elif M <= 4.5:
        return 0.2
    elif M <= 5:
        return 0.2
    elif M <= 5.5:
        return 0.2
    elif M <= 6:
        return 0.21
    elif M <= 6.5:
        return 0.23
    elif M <= 7:
        return 0.24
    elif M <= 7.5:
        return 0.25
    else:
        return 0.26

def mach(v, y):
    gamma = 1.4
    R = 287

    c = np.sqrt(gamma * R * T(y))

    return abs(v) / c


def equations(t, state):
    y, v = state

    dydt = v
    M = mach(v, y)

    dvdt = -g - (Cd(M) * rho(y) * A * v * abs(v)) / (2 * m)

    return [dydt, dvdt]



y = 0
v = 2100.5
t = 0

dt = 0.5

while v > 0:

    c = np.sqrt(1.4 * 287 * T(y))

    M = v / c

    F_drag = 0.5 * Cd(M) * rho(y) * A * v**2

    a = -g - F_drag / m

    y = y + v * dt

    v = v + a * dt

    t = t + dt

    print("Time:", t, "s")
    print("Altitude:", y / 1000, "km")
    print("Velocity:", v, "m/s")
    print("--------------------")

Time: 0.5 s
Altitude: 1.05025 km
Velocity: 1946.3612011944813 m/s
--------------------
Time: 1.0 s
Altitude: 2.0234306005972407 km
Velocity: 1836.471709773595 m/s
--------------------
Time: 1.5 s
Altitude: 2.941666455484038 km
Velocity: 1750.887261287686 m/s
--------------------
Time: 2.0 s
Altitude: 3.817110086127881 km
Velocity: 1679.0768726328017 m/s
--------------------
Time: 2.5 s
Altitude: 4.656648522444282 km
Velocity: 1617.7821448449909 m/s
--------------------
Time: 3.0 s
Altitude: 5.465539594866778 km
Velocity: 1564.7458339007878 m/s
--------------------
Time: 3.5 s
Altitude: 6.247912511817171 km
Velocity: 1518.344594143787 m/s
--------------------
Time: 4.0 s
Altitude: 7.007084808889065 km
Velocity: 1477.374472152484 m/s
--------------------
Time: 4.5 s
Altitude: 7.745772044965307 km
Velocity: 1440.9192076142085 m/s
--------------------
Time: 5.0 s
Altitude: 8.46623164877241 km
Velocity: 1408.2659812477314 m/s
--------------------
Time: 5.5 s
Altitude: 9.170364639396277 km
V